In [1]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import glob
import os
import netCDF4 as nc4
from scipy.interpolate import interp1d
import math
import matplotlib.tri as tri
#from plot_season import make_hist
import pickle as pkl
import sim_name_remap as map
import plotter_config as pc
import jdcal

In [2]:

sim_names = ['2022_M2-3hr','2022_E5-3hr','Aug2022_E5-1hr','Aug2022_E5-1hr_oxSO2update','Aug2022_E5-1hr_oxSO2-0.75','Aug2022_E5-1hr_noH2SO4prod','2022_E5-1hr_H2SO4prod-0.75']

var_names = ['SO2','SO4']#'CO','E90','SO2',]

#lon_bndss = [[0,360]]
#lat_bndss = [[-20,20]]
#lon_bndss = [[123,126]]
#lat_bndss = [[33,38]]
lon_bndss = [[125,140]]
lat_bndss = [[25,38]]

date0 = '2022-08-01'
date1 = '2022-08-31'

SAVEDIR = '/glade/u/home/wsmith/ASM_Plotter/vars/'

z_prof = np.arange(0,25.1,0.5)  # Altitude profile (km)


In [3]:
for sim_name in sim_names:

    true_sim_path = map.remap[sim_name]

    print(sim_name)

    FILES = sorted(glob.glob(true_sim_path))

    FILES0 = [item for item in FILES if date0 in item][0]
    find0 = list(FILES).index(FILES0)
    FILES1 = [item for item in FILES if date1 in item][0]
    find1 = list(FILES).index(FILES1)
    
    FILES = FILES[find0:find1+1]
    
    for ltln in range(len(lon_bndss)):

        lon_bnds = lon_bndss[ltln]
        lat_bnds = lat_bndss[ltln]
                   
        for v in range(len(var_names)):
                            
            var_name = var_names[v]
    
            zSAVEFILE = SAVEDIR + var_name + 'zprofiles_' + sim_name + '_' + str(lon_bnds) + '_' + str(lat_bnds) + '.pkl'  # + '_' + date0 + '-' + date1 +   This needs to disappear right now because I have a talk deadline
    
            if os.path.isfile(zSAVEFILE):
                print(var_name, ' files already exist, skipping them...')
                
            else: 
    
                var_str = pc.var_info[var_name]['vars']
        
                print(sim_name, '*****', var_str, '*****', lon_bnds, '*****', lat_bnds, '*****', date0 + '_' + date1)
    
                var_zprof = np.zeros((1,len(z_prof)))-999
                jds = []
    
                for FILE in FILES:
                    print(FILE)
    
                    file = nc4.Dataset(FILE)
    
                    lon  = file.variables['lon'][:]
                    lat  = file.variables['lat'][:]
                    lev  = file.variables['lev'][:]
                    hyam = file.variables['hyam'][:]
                    hybm = file.variables['hybm'][:]
                    PS     = file.variables['PS'][:]
                    gph    = file.variables['Z3'][:]
                    date   = file.variables['date'][:]
                    datesec = file.variables['datesec'][:]                    
        
                    # Account for the fact that we might have two model variables that we need to combine
                    if isinstance(var_str, str): var = file.variables[var_str][:]
                    if isinstance(var_str, list):
                        var = file.variables[var_str[0]][:]
                        for sub_v in range(len(var_str)-1):
                            sub_var = file.variables[var_str[sub_v+1]][:]
                            var = var + sub_var
    
                    NT = np.shape(var)[0]
                    
                    PS_mean = np.mean(PS, axis=0)
    
                    # Determine indices of the boundaries
                    lon_ind0 = np.min(np.where(lon >= lon_bnds[0])[0])
                    lon_ind1 = np.max(np.where(lon <= lon_bnds[1])[0])+1
                    lat_ind0 = np.min(np.where(lat >= lat_bnds[0])[0])
                    lat_ind1 = np.max(np.where(lat <= lat_bnds[1])[0])+1
    
                    # Trim the arrays to just the lon/lat boundaries
                    var_sub = var[:,:,lat_ind0:lat_ind1,lon_ind0:lon_ind1]
                    gph_sub = gph[:,:,lat_ind0:lat_ind1,lon_ind0:lon_ind1]
                    PS_mean_sub = PS_mean[lat_ind0:lat_ind1,lon_ind0:lon_ind1]
                    lon_sub = lon[lon_ind0:lon_ind1]
                    lat_sub = lat[lat_ind0:lat_ind1]
        
                    ################ Compute average profiles within and outside the anticyclone
                    for t in range(NT):
                        curr_var_zprof = np.zeros((1,len(z_prof)))-999
                        #print('****************  t = ', t)                        
                        for lt in range(len(lat_sub)):
                            for ln in range(len(lon_sub)):                                    
    
                                # Unfortunately these simulations are on "raw" model levels which means we need to account for topography.                                  
                                curr_PS = PS_mean_sub[lt,ln]
                                curr_plev = (hyam*1.0e5 + hybm*curr_PS)/100.
                                curr_gphprof = np.squeeze(gph_sub[t,:,lt,ln])                            
                                curr_varprof = np.squeeze(var_sub[t,:,lt,ln])
                                
                                f_z = interp1d(curr_gphprof*1.e-3, curr_varprof, bounds_error=False, fill_value=-999)
                                
                                curr_var_zprof = np.append(curr_var_zprof, f_z(z_prof)[None,:], axis=0)
                               
                        jds.append(sum(jdcal.gcal2jd(str(date[t])[0:4], str(date[t])[4:6], str(date[t])[6:8])) + datesec[t]/86400.) 
                        var_zprof = np.append(var_zprof, np.mean(curr_var_zprof[1:,:], axis=0)[None,:], axis=0)
                        
                    ################# Save the profile information
    
                print('Shape of var_zprof array being saved = ', np.shape(var_zprof[1:,:]))
    
                print(np.shape(var_zprof[1:,:]))
                print(np.shape(jds))
                
                with open(zSAVEFILE, 'wb') as f:
                    pkl.dump((var_zprof[1:,:], z_prof, jds), f)
        

                            

2022_M2-3hr
SO2  files already exist, skipping them...
SO4  files already exist, skipping them...
2022_E5-3hr
SO2  files already exist, skipping them...
SO4  files already exist, skipping them...
Aug2022_E5-1hr
SO2  files already exist, skipping them...
SO4  files already exist, skipping them...
Aug2022_E5-1hr_oxSO2update
SO2  files already exist, skipping them...
SO4  files already exist, skipping them...
Aug2022_E5-1hr_oxSO2-0.75
SO2  files already exist, skipping them...
SO4  files already exist, skipping them...
Aug2022_E5-1hr_noH2SO4prod
SO2  files already exist, skipping them...
SO4  files already exist, skipping them...
2022_E5-1hr_H2SO4prod-0.75
2022_E5-1hr_H2SO4prod-0.75 ***** ['SO2'] ***** [125, 140] ***** [25, 38] ***** 2022-08-01_2022-08-31
/glade/derecho/scratch/wsmith/archive/FCnudged_f09.carmats16_cesm2.2_2022_E5-1hr_H2SO4prod-0.75/atm/hist/FCnudged_f09.carmats16_cesm2.2_2022_E5-1hr_H2SO4prod-0.75.cam.h1.2022-08-01-10800.nc
/glade/derecho/scratch/wsmith/archive/FCnudged_